# Задание 3. Расчет коэффициента D Жуйана

Коэффициент Жуйана характеризует равномерность распределения слова по сегментам корпуса:

$$D = 1 - \frac{\sigma}{\mu \sqrt{n-1}},$$

где $\mu$ — средняя частота слова в сегменте, $\sigma$ — среднее квадратическое отклонение частоты по сегментам (вычисляется с делением на $n$), $n$ — число сегментов.

Чем ближе $D$ к 1, тем равномернее слово распределено по корпусу. Для служебных слов значение $D$ обычно высокое, для редких и тематически связанных слов — низкое.

Корпус разбивается на 4 равных сегмента по 1,25 млн словоупотреблений. Чтобы не хранить весь корпус, в каждом сегменте подсчитываются частоты только слов из топ-1000 соответствующего частотного словаря. Результаты кэшируются в `rgr2/data/segments_ru.json` и `rgr2/data/segments_en.json`.


In [1]:
from pathlib import Path
import json
import re
import time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'freq_ru.json').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'rgr2'
DATA = BASE / 'data'
FIG = BASE / 'figures'
DATA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

DICTIONARIES = [
    ('ru_raw',   'Русский, без лемматизации',  'freq_ru.json',       'ru', 'raw'),
    ('ru_lemma', 'Русский, с лемматизацией',   'freq_ru_lemma.json', 'ru', 'lemma'),
    ('en_raw',   'Английский, без лемматизации', 'freq_en.json',     'en', 'raw'),
    ('en_lemma', 'Английский, с лемматизацией',  'freq_en_lemma.json', 'en', 'lemma'),
]

TARGET_TOKENS = 5_000_000
N_SEGMENTS = 4
SEGMENT_SIZE = TARGET_TOKENS // N_SEGMENTS
TOP_N = 1000

PATTERNS = {
    'ru': re.compile(r'[а-яё]+', re.IGNORECASE),
    'en': re.compile(r'[a-z]+', re.IGNORECASE),
}

def load_dictionary(filename):
    with open(ROOT / filename, encoding='utf-8') as f:
        package = json.load(f)
    items = sorted(package['dictionary'].items(), key=lambda p: (-p[1], p[0]))
    df = pd.DataFrame(items, columns=['word', 'freq'])
    df['rank'] = np.arange(1, len(df) + 1)
    return package, df

packages = {}
datasets = {}
for key, label, filename, lang, variant in DICTIONARIES:
    package, df = load_dictionary(filename)
    packages[key] = package
    datasets[key] = {'df': df, 'label': label, 'lang': lang, 'variant': variant, 'file': filename}
    print(f"{label}: токенов {package['total_tokens']:,}, топ-{TOP_N} подготовлен")


Русский, без лемматизации: токенов 4,982,552, топ-1000 подготовлен


Русский, с лемматизацией: токенов 5,000,251, топ-1000 подготовлен


Английский, без лемматизации: токенов 5,002,249, топ-1000 подготовлен


Английский, с лемматизацией: токенов 5,002,249, топ-1000 подготовлен


In [2]:
targets = {'ru': {'raw': set(), 'lemma': set()}, 'en': {'raw': set(), 'lemma': set()}}
for key, label, filename, lang, variant in DICTIONARIES:
    targets[lang][variant] = set(datasets[key]['df'].head(TOP_N)['word'])
    print(f"{label}: целевых слов {len(targets[lang][variant])}")


Русский, без лемматизации: целевых слов 1000
Русский, с лемматизацией: целевых слов 1000
Английский, без лемматизации: целевых слов 1000
Английский, с лемматизацией: целевых слов 1000


In [3]:
def build_segment_counts(lang):
    """Считает частоты целевых слов в 4 сегментах корпуса и кэширует результат."""
    cache_path = DATA / f'segments_{lang}.json'
    version = 1

    if cache_path.exists():
        with open(cache_path, encoding='utf-8') as f:
            cached = json.load(f)
        if cached.get('version') == version and cached.get('target_tokens') == TARGET_TOKENS:
            print(f'{lang}: загружен кэш {cache_path.name} '
                  f"(токенов {cached['actual_tokens']:,})")
            return cached

    from datasets import load_dataset
    import simplemma

    target_raw = targets[lang]['raw']
    target_lemma = targets[lang]['lemma']
    pattern = PATTERNS[lang]

    raw_counts = [Counter() for _ in range(N_SEGMENTS)]
    lemma_counts = [Counter() for _ in range(N_SEGMENTS)]
    segment_totals = [0] * N_SEGMENTS

    dataset = load_dataset('wikimedia/wikipedia', name=f'20231101.{lang}',
                           split='train', streaming=True)

    total = 0
    next_report = 0
    started = time.time()
    articles = 0

    for article in dataset:
        articles += 1
        for token in pattern.findall(article['text']):
            if total >= TARGET_TOKENS:
                break
            word = token.lower()
            segment = min(total // SEGMENT_SIZE, N_SEGMENTS - 1)

            if word in target_raw:
                raw_counts[segment][word] += 1

            lemma = simplemma.lemmatize(word, lang=lang)
            if lemma in target_lemma:
                lemma_counts[segment][lemma] += 1

            segment_totals[segment] += 1
            total += 1

            if total >= next_report:
                elapsed = time.time() - started
                print(f'{lang}: {total:,} / {TARGET_TOKENS:,} токенов, '
                      f'статей {articles:,}, {elapsed:.0f} с')
                next_report += 500_000

        if total >= TARGET_TOKENS:
            break

    payload = {
        'version': version,
        'lang': lang,
        'target_tokens': TARGET_TOKENS,
        'actual_tokens': int(total),
        'n_segments': N_SEGMENTS,
        'segment_totals': segment_totals,
        'top_n': TOP_N,
        'raw': {w: [int(raw_counts[s].get(w, 0)) for s in range(N_SEGMENTS)]
                for w in sorted(target_raw)},
        'lemma': {w: [int(lemma_counts[s].get(w, 0)) for s in range(N_SEGMENTS)]
                  for w in sorted(target_lemma)},
    }

    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False)
    print(f'{lang}: сохранено в {cache_path} ({cache_path.stat().st_size / 1024:.1f} КБ)')
    return payload

segments = {}
for lang in ['ru', 'en']:
    segments[lang] = build_segment_counts(lang)
    print('Сегменты:', segments[lang]['segment_totals'])


Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

ru: 1 / 5,000,000 токенов, статей 1, 7 с


ru: 500,000 / 5,000,000 токенов, статей 119, 9 с


ru: 1,000,000 / 5,000,000 токенов, статей 410, 11 с


ru: 1,500,000 / 5,000,000 токенов, статей 616, 14 с


ru: 2,000,000 / 5,000,000 токенов, статей 854, 15 с


... (14 outputs left out)

en: 3,500,000 / 5,000,000 токенов, статей 977, 15 с


en: 4,000,000 / 5,000,000 токенов, статей 1,195, 16 с


en: 4,500,000 / 5,000,000 токенов, статей 1,369, 16 с


en: 5,000,000 / 5,000,000 токенов, статей 1,485, 17 с
en: сохранено в /home/dd/nlp_uni/rgr2/data/segments_en.json (62.8 КБ)
Сегменты: [1250000, 1250000, 1250000, 1250000]


In [4]:
def juilland_stats(counts):
    x = np.asarray(counts, dtype=float)
    n = len(x)
    mu = float(x.mean())
    sigma = float(x.std(ddof=0))
    if mu <= 0 or n < 2:
        return np.nan, mu, sigma
    d = 1.0 - sigma / (mu * np.sqrt(n - 1))
    return float(d), mu, sigma

rows = []
for key, label, filename, lang, variant in DICTIONARIES:
    df = datasets[key]['df'].head(TOP_N)
    counts_map = segments[lang][variant]
    total_tokens = packages[key]['total_tokens']
    for _, row in df.iterrows():
        word = row['word']
        counts = counts_map.get(word, [0] * N_SEGMENTS)
        d, mu, sigma = juilland_stats(counts)
        rows.append({
            'key': key,
            'dictionary': label,
            'lang': lang,
            'variant': variant,
            'word': word,
            'freq': int(row['freq']),
            'rel_freq_per_1e6': row['freq'] / total_tokens * 1e6,
            'rank': int(row['rank']),
            'segments': counts,
            'mu_segment': mu,
            'sigma_segment': sigma,
            'D': d,
        })

juilland = pd.DataFrame(rows)
juilland.to_csv(DATA / 'juilland_top1000.csv', index=False, float_format='%.8f')

table = juilland.groupby('key', group_keys=False).head(10).copy()
table.to_csv(DATA / 'juilland_table_top10.csv', index=False, float_format='%.8f')

def rank_band(rank):
    if rank <= 10:
        return '1–10'
    if rank <= 100:
        return '11–100'
    return '101–1000'

juilland['band'] = juilland['rank'].apply(rank_band)
band_order = ['1–10', '11–100', '101–1000']
juilland['band'] = pd.Categorical(juilland['band'], categories=band_order, ordered=True)
bands = (juilland.groupby(['key', 'dictionary', 'band'], observed=True)
         .agg(n_words=('D', 'size'), mean_D=('D', 'mean'), median_D=('D', 'median'),
              min_D=('D', 'min'), max_D=('D', 'max'))
         .reset_index())
bands.to_csv(DATA / 'juilland_rank_bands.csv', index=False, float_format='%.8f')

display(table[['dictionary', 'word', 'freq', 'rel_freq_per_1e6', 'rank', 'D']]
        .rename(columns={
            'dictionary': 'Словарь', 'word': 'Слово', 'freq': 'Частота',
            'rel_freq_per_1e6': 'Отн. частота (ipm)', 'rank': 'Ранг', 'D': 'D',
        }).round(3))
print('\nСредний D по частотным зонам:')
display(bands[['dictionary', 'band', 'n_words', 'mean_D', 'median_D', 'min_D', 'max_D']]
        .rename(columns={
            'dictionary': 'Словарь', 'band': 'Ранги', 'n_words': 'Слов',
            'mean_D': 'Средний D', 'median_D': 'Медиана D',
            'min_D': 'Мин. D', 'max_D': 'Макс. D',
        }).round(3))


,Словарь,Слово,Частота,Отн. частота (ipm),Ранг,D
0,"Русский, без лемматизации",в,230296,46220.491,1,0.977
1,"Русский, без лемматизации",и,172762,34673.396,2,0.991
2,"Русский, без лемматизации",на,71193,14288.461,3,0.970
3,"Русский, без лемматизации",с,58594,11759.837,4,0.981
4,"Русский, без лемматизации",по,32946,6612.274,5,0.962
5,"Русский, без лемматизации",из,27528,5524.880,6,0.981
6,"Русский, без лемматизации",года,25444,5106.620,7,0.947
7,"Русский, без лемматизации",к,25213,5060.258,8,0.970
8,"Русский, без лемматизации",не,23898,4796.337,9,0.961
9,"Русский, без лемматизации",году,23173,4650.830,10,0.973



Средний D по частотным зонам:


,Словарь,Ранги,Слов,Средний D,Медиана D,Мин. D,Макс. D
0,"Английский, с лемматизацией",1–10,10,0.985,0.985,0.975,0.997
1,"Английский, с лемматизацией",11–100,90,0.945,0.962,0.703,0.993
2,"Английский, с лемматизацией",101–1000,900,0.880,0.908,0.245,0.996
3,"Английский, без лемматизации",1–10,10,0.980,0.985,0.951,0.997
4,"Английский, без лемматизации",11–100,90,0.947,0.965,0.703,0.993
5,"Английский, без лемматизации",101–1000,900,0.877,0.908,0.053,0.987
6,"Русский, с лемматизацией",1–10,10,0.974,0.974,0.960,0.991
7,"Русский, с лемматизацией",11–100,90,0.910,0.940,0.602,0.995
8,"Русский, с лемматизацией",101–1000,900,0.870,0.900,0.186,0.993
9,"Русский, без лемматизации",1–10,10,0.971,0.972,0.947,0.991


In [5]:
def make_juilland_figure(keys, title, out_path):
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    colors = {'ru_raw': '#1f77b4', 'ru_lemma': '#d62728',
              'en_raw': '#1f77b4', 'en_lemma': '#d62728'}
    for key in keys:
        df = juilland[juilland['key'] == key].sort_values('rank')
        label = datasets[key]['label']
        ax.scatter(df['rank'], df['D'], s=7, alpha=0.28,
                   color=colors[key], linewidths=0)
        smooth = (df.set_index('rank')['D']
                  .rolling(window=31, center=True, min_periods=5).median())
        ax.plot(smooth.index, smooth.values, color=colors[key], linewidth=1.8,
                label=label)
    ax.set_xscale('log')
    ax.set_xlabel('Ранг слова r')
    ax.set_ylabel('Коэффициент D Жуйана')
    ax.set_ylim(-0.5, 1.05)
    ax.set_title(title)
    ax.legend(loc='lower left')
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print('Сохранено:', out_path)

make_juilland_figure(['ru_raw', 'ru_lemma'],
                     'Коэффициент D Жуйана для русского языка',
                     FIG / 'fig_juilland_ru.png')
make_juilland_figure(['en_raw', 'en_lemma'],
                     'Коэффициент D Жуйана для английского языка',
                     FIG / 'fig_juilland_en.png')


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_juilland_ru.png


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_juilland_en.png


In [6]:
lines = ['### Вывод по заданию 3', '']
for _, row in bands.iterrows():
    lines.append(f"- **{row['dictionary']}**, ранги {row['band']}: средний D = {row['mean_D']:.3f}, "
                 f"медиана D = {row['median_D']:.3f} (n = {int(row['n_words'])}).")
lines += [
    '',
    'Для самых частотных слов (служебная лексика, местоимения, вспомогательные глаголы) коэффициент D близок к 1: '
    'такие слова встречаются во всех сегментах корпуса примерно с одинаковой относительной частотой. '
    'С ростом ранга, то есть при переходе к менее частотной и тематически окрашенной лексике, D снижается. '
    'Это согласуется с содержательным смыслом коэффициента Жуйана: он отражает равномерность распределения слова по корпусу.',
]
display(Markdown('\n'.join(lines)))


### Вывод по заданию 3

- **Английский, с лемматизацией**, ранги 1–10: средний D = 0.985, медиана D = 0.985 (n = 10).
- **Английский, с лемматизацией**, ранги 11–100: средний D = 0.945, медиана D = 0.962 (n = 90).
- **Английский, с лемматизацией**, ранги 101–1000: средний D = 0.880, медиана D = 0.908 (n = 900).
- **Английский, без лемматизации**, ранги 1–10: средний D = 0.980, медиана D = 0.985 (n = 10).
- **Английский, без лемматизации**, ранги 11–100: средний D = 0.947, медиана D = 0.965 (n = 90).
- **Английский, без лемматизации**, ранги 101–1000: средний D = 0.877, медиана D = 0.908 (n = 900).
- **Русский, с лемматизацией**, ранги 1–10: средний D = 0.974, медиана D = 0.974 (n = 10).
- **Русский, с лемматизацией**, ранги 11–100: средний D = 0.910, медиана D = 0.940 (n = 90).
- **Русский, с лемматизацией**, ранги 101–1000: средний D = 0.870, медиана D = 0.900 (n = 900).
- **Русский, без лемматизации**, ранги 1–10: средний D = 0.971, медиана D = 0.972 (n = 10).
- **Русский, без лемматизации**, ранги 11–100: средний D = 0.908, медиана D = 0.935 (n = 90).
- **Русский, без лемматизации**, ранги 101–1000: средний D = 0.869, медиана D = 0.897 (n = 900).

Для самых частотных слов (служебная лексика, местоимения, вспомогательные глаголы) коэффициент D близок к 1: такие слова встречаются во всех сегментах корпуса примерно с одинаковой относительной частотой. С ростом ранга, то есть при переходе к менее частотной и тематически окрашенной лексике, D снижается. Это согласуется с содержательным смыслом коэффициента Жуйана: он отражает равномерность распределения слова по корпусу.